# media_flag와 action_flag 분류 노트북

이 노트북은 광고 목록 데이터에서 media_flag와 action_flag를 만드는 실습용 노트북입니다.

media_flag는 광고가 어느 매체나 플랫폼과 관련 있는지를 나타냅니다. 예를 들면 media_naver, media_youtube, media_instagram, media_app 같은 분류입니다.

action_flag는 사용자가 어떤 행동을 해야 광고가 완료되는지를 나타냅니다. 예를 들면 action_click, action_view, action_install, action_signup, action_purchase, action_participation 같은 분류입니다.

이 노트북의 핵심 방향은 BERTopic을 최종 분류기로 쓰지 않는 것입니다. BERTopic은 정답 라벨을 모르는 상태에서 비슷한 문서끼리 묶어주는 비지도 학습 도구입니다. 따라서 최종 정답을 자동으로 확정하기보다, 사람이 검수할 후보를 줄여주는 보조 도구로 사용하는 것이 안전합니다.



## 고친점 세가지

1. URL은 지우기 전에 도메인을 먼저 추출해야 합니다. youtube.com, instagram.com, naver.com 같은 도메인은 media 분류에 매우 강한 단서이기 때문.

2. action_click을 기본값으로 두면 안 됩니다. 근거가 없으면 action_unknown으로 두고, 정말 클릭이나 방문 단서가 있을 때만 action_click로 분류해야.

3. unknown이 줄었다고 성능이 좋아진 것이 아닙니다. 틀린 라벨을 많이 붙이는 것보다, 근거가 없을 때 unknown으로 남기는 것이 더 안전합니다.



## 🔄 v4 변경점 — Cascade 분류 (희소 라벨 우선)

기존 v3는 모든 라벨이 동시에 점수를 받고 argmax로 결정 → 다수 라벨(naver/participation)이 유리

**v4 cascade 구조**: 희소 라벨부터 차례로 잡고, 못 잡힌 행만 다음 단계로

### 매체 분류 순서 (희소 → 빈출)
| stage | 내용 |
|---|---|
| 1 | SNS 직접 단서 (틱톡/페북/트위터/인스타/카카오/유튜브) |
| 2 | 도메인 매칭 |
| 3 | 앱 패키지/앱 명시 → media_app |
| 4 | 웹 단서 (회원가입/구매/체험) |
| 5 | 네이버 명시 단서 (직접 키워드) |
| 6 | 네이버 정황 단서 (퀴즈/플레이스/스크린샷) |
| 7 | ads_type 강제 매핑 (보강) |
| 8 | unknown |

### 행동 분류 순서 (참여형 마지막)
| stage | 내용 |
|---|---|
| 1 | purchase/signup/install/run/view/exposure/click 키워드 |
| 2 | ads_type 1/2/4/8/12 강제 매핑 (참여형 제외) |
| 3 | ads_type=3 → participation (마지막) |
| 4 | unknown |

### 효과
- SNS/카카오/시청형/가입형 등 소수 라벨이 다수 라벨에 흡수되지 않음
- `rule_media_stage`, `rule_action_stage` 컬럼 추가 — 어느 단계에서 결정됐는지 추적 가능



## 📌 v3 최적화 변경점 (이전 점검 기반)

이전 점검에서 확인된 7가지 핵심 문제를 수정했습니다. **셀 구조와 변수명은 그대로** 유지했고, 셀 단위로 핵심 로직만 개선했습니다.

| # | 문제 (이전) | 수정 (v3) | 셀 |
|---|---|---|---|
| 1 | `ads_type=1`(설치) 78%가 잘못 분류됨 | `ads_type` 강제 매핑을 룰 최우선 신호(+5.0)로 적용 | 17, 19 |
| 2 | `action_exposure`가 결과 0건인데 토픽엔 2066건 best | exposure 시드 단순화 (사실상 view로 흡수) | 15 |
| 3 | 토픽 score margin 99.8%가 <0.1 (변별력 X) | 시드 다양화 + 임계값 현실화 (0.38→0.50) | 15, 33 |
| 4 | media_unknown 70K (퀴즈 광고) | `media_naver`에 save_way 빈출 표현 추가 + save_way 다수결 후처리 | 15, 35 |
| 5 | `action_click` 키워드 너무 넓음 (사이트 방문/접속) | "사이트 방문/접속/페이지 방문/랜딩" 제거 | 15 |
| 6 | 라벨 간 cross-contamination | `negative_keywords` 도입 (-1.5 차감) | 15, 17, 19 |
| 7 | `apply(axis=1)` 메모리 폭발 | 모든 룰을 numpy 벡터화 | 13, 17, 19, 35 |

**예상 효과**:
- `ads_type=1/2/4` 정합성 21~30% → 95%+ (강제 매핑이 텍스트 매칭을 압도)
- `media_unknown` 70K → 1만 미만 (save_way 다수결로 흡수)
- 토픽 high_confidence 표기 신뢰도 회복 (margin 임계 5%)
- 실행 속도/메모리 약 5~10배 개선 (벡터화)


## 0. 실습 전에 이해해야 할 용어

rule-based 분류는 사람이 직접 만든 규칙으로 라벨을 붙이는 방식입니다. 예를 들어 문장에 유튜브, 쇼츠, 영상시청이 있으면 media_youtube 후보로 보는 방식입니다. 빠르고 설명 가능하지만, 사람이 생각하지 못한 표현은 놓칠 수 있습니다.

embedding은 텍스트를 숫자 벡터로 바꾸는 과정입니다. 컴퓨터는 문장을 그대로 이해하지 못하므로 문장의 의미를 숫자 배열로 바꿉니다. 의미가 비슷한 문장들은 비슷한 벡터를 갖도록 만드는 것이 목표입니다.

BERTopic은 embedding으로 바꾼 문서들을 비슷한 것끼리 묶어서 topic을 만드는 방법입니다. 여기서 topic은 정답 라벨이 아니라, 비슷한 광고 문구들의 묶음입니다.

confidence는 이 라벨을 얼마나 믿을 수 있는지를 나타내는 값입니다. 이 노트북에서는 점수와 evidence, 즉 근거 단어를 같이 저장합니다.

validation set은 사람이 직접 정답을 붙인 검수용 데이터입니다. 모델이나 규칙이 잘 맞는지 확인하려면 반드시 사람이 붙인 정답이 필요합니다.


## 1. 설치 셀

처음 실행하는 환경이라면 아래 코드 셀에서 pip install 주석을 풀고 실행하세요.

Colab에서는 설치가 끝난 뒤 런타임을 재시작해야 하는 경우가 있습니다. 특히 hdbscan, umap-learn 관련 오류가 나면 런타임 재시작 후 다시 실행하세요.

이미 설치된 환경에서는 이 셀을 실행하지 않아도 됩니다.


In [1]:
# VSCode 환경에서는 아래 셀을 실행하지 말고,
# 터미널에서 직접 설치하세요:
#
#   pip install pandas openpyxl numpy tqdm scikit-learn sentence-transformers bertopic umap-learn hdbscan
#
# 이미 설치된 환경이면 이 셀은 그냥 넘어가세요.

# !pip install -q pandas openpyxl numpy tqdm scikit-learn sentence-transformers bertopic umap-learn hdbscan
print('패키지 설치 확인 완료 (터미널에서 설치 필요 시 위 주석 참고)')


패키지 설치 확인 완료 (터미널에서 설치 필요 시 위 주석 참고)


## 2. 라이브러리 불러오기와 경로 설정

이 셀에서는 필요한 라이브러리를 불러오고, 입력 파일과 결과 저장 폴더를 설정합니다.



In [2]:
import os
import re
import json
import math
import warnings
import unicodedata
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# 입력 파일 경로 - 실제 파일 위치에 맞게 수정하세요
# parquet 파일을 사용하므로 ADS_FILE은 참고용 (실제 로드는 셀 9에서 parquet으로 함)
ADS_FILE = Path(r'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/parquet_debug/IVE_광고목록__Result 1.parquet')

# 결과 저장 폴더
OUT_DIR = Path(r'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/media_action_flag_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('입력 파일:', ADS_FILE)
print('결과 저장 폴더:', OUT_DIR)
print('입력 파일 존재 여부:', ADS_FILE.exists())


입력 파일: C:\Users\hoo58\OneDrive\바탕 화면\tables\DATA\parquet_debug\IVE_광고목록__Result 1.parquet
결과 저장 폴더: C:\Users\hoo58\OneDrive\바탕 화면\tables\DATA\media_action_flag_outputs
입력 파일 존재 여부: True


C:\Users\hoo58\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 데이터 불러오기

엑셀 파일을 pandas DataFrame으로 읽어옵니다.

DataFrame은 행과 열로 이루어진 표 형태 데이터입니다. 엑셀 시트와 비슷하다고 생각하면 됩니다.

데이터를 불러온 뒤에는 반드시 shape와 컬럼명을 확인해야 합니다. shape는 행 개수와 열 개수를 의미합니다. 예를 들어 9000행 30열이면 shape가 (9000, 30)처럼 나옵니다.


In [3]:
import polars as pl
import pandas as pd

# ADS_FILE은 셀 7에서 설정한 경로를 그대로 사용합니다
df_pl = pl.read_parquet(str(ADS_FILE))
df = df_pl.to_pandas()

print('데이터 크기:', df.shape)
print('컬럼 목록:', df.columns.tolist())

# 전체 실행 전 소규모 테스트가 필요하면 아래 주석을 풀고 실행하세요
# df = df.sample(n=5000, random_state=RANDOM_STATE).reset_index(drop=True)
# print('샘플링 후 크기:', df.shape)


데이터 크기: (445260, 30)
컬럼 목록: ['ads_idx', 'ads_code', 'aff_idx', 'adv_idx', 'sch_idx', 'ads_type', 'ads_category', 'ads_name', 'ads_search', 'ads_icon_img', 'ads_summary', 'ads_guide', 'ads_limit', 'ads_payment', 'ads_save_way', 'ads_day_cap', 'ads_sdate', 'ads_edate', 'ads_package', 'ads_sex_type', 'ads_age_min', 'ads_age_max', 'ads_os_type', 'ads_contract_price', 'ads_reward_price', 'ads_order', 'ads_rejoin_type', 'ads_require_adid', 'regdate', 'delyn']


## 4. 분류에 사용할 컬럼 고르기

광고 데이터에는 숫자 컬럼, 날짜 컬럼, 텍스트 컬럼이 섞여 있습니다.

media와 action을 분류할 때는 모든 컬럼이 똑같이 중요하지 않습니다. 예를 들어 ads_name, ads_summary, ads_guide, ads_save_way, ads_search 같은 컬럼은 광고 내용을 설명하므로 중요합니다.

반대로 ads_idx, adv_idx 같은 ID 컬럼은 보통 라벨 분류에 직접적인 의미가 적습니다.

이 셀에서는 강한 텍스트 컬럼과 약한 텍스트 컬럼을 나눕니다.

강한 텍스트 컬럼은 광고 미션이나 분류에 직접 관련된 컬럼입니다. 이 컬럼에서 나온 단서는 높은 신뢰도로 봅니다.

약한 텍스트 컬럼은 아이콘 이미지 주소, 보조 링크, 설명에 섞인 URL처럼 오탐 가능성이 있는 컬럼입니다. 예를 들어 광고주 공식 홈페이지 하단에 인스타그램 링크가 있다고 해서 그 광고가 인스타그램 미션이라고 단정하면 안 됩니다.


In [4]:
# 이 목록은 IVE 광고목록 데이터 기준으로 작성했습니다.
# 다른 데이터셋에서 실행한다면 실제 컬럼명에 맞게 수정하세요.

# media와 action 분류에 강하게 사용할 컬럼입니다.
# 광고명, 요약, 가이드, 적립 방법, 검색어 등은 미션 내용을 직접 담을 가능성이 큽니다.
# 주의: ads_type은 숫자 코드이므로 키워드 매칭이 아니라 별도 룰로 처리합니다 (셀 17/19 참고).
STRONG_TEXT_COL_CANDIDATES = [
    "ads_name",
    "ads_search",
    "ads_summary",
    "ads_guide",
    "ads_limit",
    "ads_save_way",
    "ads_package",
]

# media 판단에 조심해서 사용할 컬럼입니다.
# 이미지 URL이나 부가 URL은 광고 미션이 아니라 소재 저장 위치일 수 있으므로 weak로 둡니다.
WEAK_TEXT_COL_CANDIDATES = [
    "ads_icon_img",
]

# ads_type, ads_category는 숫자 코드 컬럼이므로 룰에서 직접 참조합니다.
# 1=설치, 2=실행, 3=참여, 4=클릭, 5=페북, 6=트위터, 7=인스타,
# 8=노출, 9=퀘스트, 10=유튜브, 11=네이버, 12=CPS(구매)
ADS_TYPE_COL = "ads_type" if "ads_type" in df.columns else None

# 실제 데이터에 존재하는 컬럼만 남깁니다.
strong_text_cols = [c for c in STRONG_TEXT_COL_CANDIDATES if c in df.columns]
weak_text_cols = [c for c in WEAK_TEXT_COL_CANDIDATES if c in df.columns]

print("강한 텍스트 컬럼:", strong_text_cols)
print("약한 텍스트 컬럼:", weak_text_cols)
print("ads_type 컬럼 존재:", ADS_TYPE_COL is not None)

# 결측치를 빈 문자열로 바꿔둡니다.
for col in strong_text_cols + weak_text_cols:
    df[col] = df[col].fillna("").astype(str)


강한 텍스트 컬럼: ['ads_name', 'ads_search', 'ads_summary', 'ads_guide', 'ads_limit', 'ads_save_way', 'ads_package']
약한 텍스트 컬럼: ['ads_icon_img']
ads_type 컬럼 존재: True


## 5. URL, 도메인, 앱 패키지 추출

이 노트북에서 가장 중요한 개선점입니다.

보통 텍스트 전처리에서는 URL을 제거합니다. 하지만 이 프로젝트에서는 URL이 매우 중요한 정보입니다. 예를 들어 youtube.com이 있으면 media_youtube의 강한 근거가 될 수 있고, play.google.com이 있으면 media_app의 강한 근거가 될 수 있습니다.

따라서 URL을 지우기 전에 도메인을 먼저 추출합니다.

또한 ads_package 같은 컬럼에 com.company.app 형태의 앱 패키지명이 있으면 앱 광고일 가능성이 높습니다. 이것도 media_app의 강한 근거로 사용합니다.


In [5]:
# URL을 찾기 위한 정규표현식입니다.
URL_PATTERN = re.compile(
    r"(https?://[^\s\]\)\}<>\"\']+|www\.[^\s\]\)\}<>\"\']+)",
    flags=re.IGNORECASE,
)

# 앱 패키지명을 찾기 위한 정규표현식입니다.
# 예: com.company.app, kr.co.service.app
APP_PACKAGE_PATTERN = re.compile(
    r"\b[a-zA-Z][a-zA-Z0-9_]*(?:\.[a-zA-Z][a-zA-Z0-9_]*){2,}\b"
)

def normalize_text_basic(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_urls(text):
    text = normalize_text_basic(text)
    return URL_PATTERN.findall(text)

def normalize_domain(domain):
    domain = str(domain).lower().strip()
    # m., mobile., kr. 같은 흔한 서브도메인 정규화
    for prefix in ("www.", "m.", "mobile.", "kr."):
        if domain.startswith(prefix):
            domain = domain[len(prefix):]
            break
    domain = domain.split(":")[0]
    return domain

def extract_domains(text):
    domains = []
    for url in extract_urls(text):
        parse_url = url
        if parse_url.startswith("www."):
            parse_url = "https://" + parse_url
        try:
            parsed = urlparse(parse_url)
            domain = normalize_domain(parsed.netloc)
            if domain:
                domains.append(domain)
        except Exception:
            pass
    return sorted(set(domains))

def extract_app_packages(text):
    text = normalize_text_basic(text)
    packages = APP_PACKAGE_PATTERN.findall(text)
    filtered = []
    for p in packages:
        if p.count(".") >= 2:
            # 흔한 false positive 제거 (도메인을 패키지로 잡는 경우)
            if any(p.endswith(suf) for suf in (".com", ".kr", ".net", ".co.kr", ".io")):
                continue
            filtered.append(p)
    return sorted(set(filtered))

def clean_text_for_embedding(text):
    text = normalize_text_basic(text)
    text = URL_PATTERN.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


## 6. 강한 단서와 약한 단서 문서 만들기

하나의 광고 행에는 여러 컬럼이 있습니다. BERTopic이나 rule 분류에 사용하려면 여러 컬럼을 하나의 문서처럼 합쳐주는 것이 편합니다.

여기서는 세 가지 문서를 만듭니다.

strong_text는 광고 미션과 직접 관련된 컬럼을 합친 텍스트입니다.

weak_text는 이미지 URL처럼 참고는 할 수 있지만 오탐 가능성이 있는 텍스트입니다.

classification_doc은 strong_text, 도메인, 앱 패키지 정보를 합친 최종 분석용 문서입니다.

중요한 점은 도메인 정보를 별도로 넣어준다는 것입니다. 그냥 URL을 지우면 youtube.com 같은 결정적인 단서가 사라집니다.


In [6]:
# 텍스트 컬럼을 벡터화로 결합합니다 (apply(axis=1)보다 빠르고 메모리 효율적입니다).
def join_columns_vectorized(frame, cols):
    if not cols:
        return pd.Series([""] * len(frame), index=frame.index)
    parts = []
    for col in cols:
        prefix = f"{col}: "
        # 정규화는 한 번에
        s = frame[col].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip().str.lower()
        # NFKC 정규화는 벡터 연산이 안 되므로 map으로 (빈 문자열은 건너뜀)
        s = s.map(lambda x: unicodedata.normalize("NFKC", x) if x else "")
        prefixed = (prefix + s).where(s != "", "")
        parts.append(prefixed)
    out = parts[0]
    for p in parts[1:]:
        # 빈 칸은 구분자 없이 합쳐야 하므로 조건부 결합
        sep = pd.Series([" | "] * len(out), index=out.index)
        sep = sep.where((out != "") & (p != ""), "")
        out = out + sep + p
    return out

df["strong_text_raw"] = join_columns_vectorized(df, strong_text_cols)
df["weak_text_raw"] = join_columns_vectorized(df, weak_text_cols)

# 도메인/패키지 추출 (이쪽은 정규식이 필수라 map으로)
df["strong_domains"] = df["strong_text_raw"].map(extract_domains)
df["weak_domains"] = df["weak_text_raw"].map(extract_domains)

if "ads_package" in df.columns:
    df["app_packages_detected"] = df["ads_package"].map(extract_app_packages)
else:
    df["app_packages_detected"] = df["strong_text_raw"].map(extract_app_packages)

# embedding/룰용 텍스트 (URL 제거)
df["strong_text"] = df["strong_text_raw"].map(clean_text_for_embedding)
df["weak_text"] = df["weak_text_raw"].map(clean_text_for_embedding)

# 카운트 캐시 (이후 룰 단계에서 매번 len()을 안 부르도록)
df["_n_strong_domains"] = df["strong_domains"].str.len()
df["_n_weak_domains"] = df["weak_domains"].str.len()
df["_n_app_packages"] = df["app_packages_detected"].str.len()

def build_classification_doc_vec(frame):
    """벡터화된 classification_doc 생성. 빈 칸은 구분자도 빼서 깔끔하게 합칩니다."""
    parts = pd.Series([""] * len(frame), index=frame.index)

    def append_part(out, fragment):
        sep = pd.Series([" | "] * len(out), index=out.index)
        sep = sep.where((out != "") & (fragment != ""), "")
        return out + sep + fragment

    s_strong = ("strong_text: " + frame["strong_text"]).where(frame["strong_text"] != "", "")
    parts = append_part(parts, s_strong)

    s_sd = frame["strong_domains"].map(lambda x: "strong_domains: " + " ".join(x) if x else "")
    parts = append_part(parts, s_sd)

    s_pkg = frame["app_packages_detected"].map(lambda x: "app_package: " + " ".join(x) if x else "")
    parts = append_part(parts, s_pkg)

    s_weak = ("weak_text: " + frame["weak_text"]).where(frame["weak_text"] != "", "")
    parts = append_part(parts, s_weak)

    s_wd = frame["weak_domains"].map(lambda x: "weak_domains: " + " ".join(x) if x else "")
    parts = append_part(parts, s_wd)
    return parts

df["classification_doc"] = build_classification_doc_vec(df)
df["media_doc"] = df["classification_doc"]
df["action_doc"] = df["classification_doc"]

display(df[[
    "strong_text",
    "weak_text",
    "strong_domains",
    "weak_domains",
    "app_packages_detected",
    "classification_doc",
]].head(5))


,strong_text,weak_text,strong_domains,weak_domains,app_packages_detected,classification_doc
0,"ads_name: 리니지레드나이츠 | ads_search: 리니지레드나이츠,com....",ads_icon_img:,[facebook.com],[lh3.googleusercontent.com],[com.ncsoft.redknights],strong_text: ads_name: 리니지레드나이츠 | ads_search: ...
1,ads_name: 강철의함대:ocean overlord | ads_search: 강...,ads_icon_img:,[cafe.naver.com],[nextapps-nas.aws.appang.kr],[com.gamepub.lw.g],strong_text: ads_name: 강철의함대:ocean overlord | ...
2,"ads_name: 스노우 snow | ads_search: 스노우 snow,com....",ads_icon_img:,"[facebook.com, instagram.com, snow.me]",[nextapps-nas.aws.appang.kr],[com.campmobile.snow],strong_text: ads_name: 스노우 snow | ads_search: ...
3,ads_name: 서머너즈 워: 천공의 아레나 | ads_search: 서머너즈 워...,ads_icon_img:,"[cafe.naver.com, goo.gl, terms.withhive.com, w...",[nextapps-nas.aws.appang.kr],[com.com2us.smon.normal.freefull.google.kr.and...,strong_text: ads_name: 서머너즈 워: 천공의 아레나 | ads_s...
4,"ads_name: 하이마트 | ads_search: 하이마트,com.himart.m...",ads_icon_img:,[],[nextapps-nas.aws.appang.kr],[com.himart.main],strong_text: ads_name: 하이마트 | ads_search: 하이마트...


## 7. taxonomy 정의하기

taxonomy는 라벨과 그 라벨을 판단할 단서들을 모아둔 사전입니다.

이 노트북에서는 media taxonomy와 action taxonomy를 따로 만듭니다.

media는 광고가 어느 매체나 플랫폼과 관련 있는지를 봅니다.

action은 사용자가 무엇을 해야 하는지를 봅니다.

초보자가 많이 하는 실수는 media와 action을 섞는 것입니다. 예를 들어 구독, 팔로우, 댓글은 action 단서입니다. 이것만 보고 media_youtube나 media_instagram으로 확정하면 위험합니다. 유튜브라는 단어와 구독이 같이 있으면 youtube일 가능성이 높지만, 구독이라는 단어만으로는 어떤 매체인지 알 수 없습니다.


In [7]:
MEDIA_LABELS = [
    "media_web",
    "media_naver",
    "media_app",
    "media_youtube",
    "media_instagram",
    "media_facebook",
    "media_twitter_x",
    "media_tiktok",
    "media_kakao",
]

ACTION_LABELS = [
    "action_click",
    "action_view",
    "action_participation",
    "action_install",
    "action_run",
    "action_signup",
    "action_purchase",
    "action_exposure",
]


# 1순위: URL/domain 기반 매체 분류
# keyword보다 훨씬 신뢰도 높으므로 그대로 유지
DOMAIN_MEDIA_MAP = {
    "youtube.com": "media_youtube",
    "youtu.be": "media_youtube",

    "instagram.com": "media_instagram",

    "facebook.com": "media_facebook",
    "fb.com": "media_facebook",

    "x.com": "media_twitter_x",
    "twitter.com": "media_twitter_x",

    "tiktok.com": "media_tiktok",

    "naver.com": "media_naver",
    "blog.naver.com": "media_naver",
    "cafe.naver.com": "media_naver",
    "smartstore.naver.com": "media_naver",
    "shopping.naver.com": "media_naver",
    "map.naver.com": "media_naver",
    "place.naver.com": "media_naver",

    "kakao.com": "media_kakao",
    "pf.kakao.com": "media_kakao",
    "talk.naver.com": "media_naver",

    "play.google.com": "media_app",
    "apps.apple.com": "media_app",
    "onestore.co.kr": "media_app",
}


# 매체 taxonomy
# 원칙:
# - 매체명, 플랫폼명, 도메인성 단어만 넣음
# - 정답입력, 퀴즈, 저장, 리뷰, 팔로우, 구독 같은 행동 단어는 제외
# - media끼리 keyword 중복 없음
MEDIA_TAXONOMY = {
    "media_naver": {
        "display_name": "네이버",
        "keywords": [
            "네이버",
            "naver",
            "네이버쇼핑",
            "네이버 쇼핑",
            "스마트스토어",
            "smartstore",
            "네이버플레이스",
            "네이버 플레이스",
            "네이버지도",
            "네이버 지도",
            "네이버블로그",
            "네이버 블로그",
            "네이버카페",
            "네이버 카페",
            "네이버예약",
            "네이버 예약",
        ],
        "negative_keywords": [
            "유튜브", "youtube",
            "인스타", "instagram",
            "카카오", "kakao",
            "페이스북", "facebook",
            "틱톡", "tiktok",
            "트위터", "twitter",
        ],
        "seed_docs": [
            "네이버 쇼핑 스마트스토어 상품 페이지와 연결된 광고",
            "네이버 플레이스 업체 페이지와 연결된 광고",
            "네이버 블로그나 네이버 카페 콘텐츠와 연결된 광고",
            "네이버 지도 또는 네이버 예약 서비스와 연결된 광고",
        ],
    },

    "media_youtube": {
        "display_name": "유튜브",
        "keywords": [
            "유튜브",
            "youtube",
            "youtu.be",
            "유튜브채널",
            "유튜브 채널",
            "youtube channel",
            "쇼츠",
            "youtube shorts",
        ],
        "negative_keywords": [
            "네이버", "naver",
            "인스타", "instagram",
            "카카오", "kakao",
            "페이스북", "facebook",
            "틱톡", "tiktok",
            "트위터", "twitter",
        ],
        "seed_docs": [
            "유튜브 영상 또는 유튜브 채널과 연결된 광고",
            "유튜브 쇼츠 콘텐츠와 연결된 광고",
        ],
    },

    "media_instagram": {
        "display_name": "인스타그램",
        "keywords": [
            "인스타그램",
            "인스타",
            "instagram",
            "릴스",
            "reels",
            "인스타계정",
            "인스타 계정",
            "instagram account",
        ],
        "negative_keywords": [
            "유튜브", "youtube",
            "네이버", "naver",
            "카카오", "kakao",
            "페이스북", "facebook",
            "틱톡", "tiktok",
            "트위터", "twitter",
        ],
        "seed_docs": [
            "인스타그램 계정 또는 인스타그램 게시물과 연결된 광고",
            "인스타그램 릴스나 스토리 콘텐츠와 연결된 광고",
        ],
    },

    "media_facebook": {
        "display_name": "페이스북",
        "keywords": [
            "페이스북",
            "facebook",
            "fb.com",
            "페이스북페이지",
            "페이스북 페이지",
            "facebook page",
        ],
        "negative_keywords": [
            "인스타", "instagram",
            "유튜브", "youtube",
            "네이버", "naver",
            "카카오", "kakao",
            "틱톡", "tiktok",
            "트위터", "twitter",
        ],
        "seed_docs": [
            "페이스북 페이지 또는 페이스북 게시물과 연결된 광고",
        ],
    },

    "media_twitter_x": {
        "display_name": "트위터/X",
        "keywords": [
            "트위터",
            "twitter",
            "x.com",
            "리트윗",
            "retweet",
            "트윗",
            "tweet",
        ],
        "negative_keywords": [
            "인스타", "instagram",
            "유튜브", "youtube",
            "네이버", "naver",
            "카카오", "kakao",
            "페이스북", "facebook",
            "틱톡", "tiktok",
        ],
        "seed_docs": [
            "트위터 또는 X 게시물과 연결된 광고",
            "X 계정 또는 트윗 콘텐츠와 연결된 광고",
        ],
    },

    "media_tiktok": {
        "display_name": "틱톡",
        "keywords": [
            "틱톡",
            "tiktok",
            "틱톡계정",
            "틱톡 계정",
            "tiktok account",
        ],
        "negative_keywords": [
            "인스타", "instagram",
            "유튜브", "youtube",
            "네이버", "naver",
            "카카오", "kakao",
            "페이스북", "facebook",
            "트위터", "twitter",
        ],
        "seed_docs": [
            "틱톡 계정 또는 틱톡 영상과 연결된 광고",
        ],
    },

    "media_kakao": {
        "display_name": "카카오",
        "keywords": [
            "카카오",
            "kakao",
            "카카오톡",
            "카톡",
            "톡채널",
            "카카오채널",
            "카카오 채널",
            "플러스친구",
            "pf.kakao.com",
        ],
        "negative_keywords": [
            "네이버", "naver",
            "유튜브", "youtube",
            "인스타", "instagram",
            "페이스북", "facebook",
            "틱톡", "tiktok",
            "트위터", "twitter",
        ],
        "seed_docs": [
            "카카오톡 채널 또는 카카오 채널과 연결된 광고",
            "카카오 플러스친구 또는 톡채널과 연결된 광고",
        ],
    },

    "media_app": {
        "display_name": "앱스토어/앱마켓",
        "keywords": [
            "구글플레이",
            "google play",
            "플레이스토어",
            "play store",
            "앱스토어",
            "app store",
            "원스토어",
            "onestore",
            "앱마켓",
            "app market",
        ],
        "negative_keywords": [
            "네이버", "naver",
            "유튜브", "youtube",
            "인스타", "instagram",
            "카카오", "kakao",
            "페이스북", "facebook",
            "틱톡", "tiktok",
            "트위터", "twitter",
        ],
        "seed_docs": [
            "구글플레이 앱스토어 원스토어 같은 앱마켓과 연결된 광고",
            "모바일 앱마켓 상세 페이지와 연결된 광고",
        ],
    },

    "media_web": {
        "display_name": "일반 웹",
        "keywords": [
            "웹사이트",
            "홈페이지",
            "공식홈페이지",
            "공식 홈페이지",
            "랜딩페이지",
            "랜딩 페이지",
            "웹페이지",
            "web page",
        ],
        "negative_keywords": [
            "네이버", "naver",
            "유튜브", "youtube",
            "인스타", "instagram",
            "카카오", "kakao",
            "페이스북", "facebook",
            "틱톡", "tiktok",
            "트위터", "twitter",
            "구글플레이", "앱스토어", "원스토어",
        ],
        "seed_docs": [
            "특정 플랫폼이 아닌 일반 웹사이트나 홈페이지와 연결된 광고",
            "브랜드 랜딩페이지 또는 웹페이지와 연결된 광고",
        ],
    },
}


# action taxonomy
# 원칙:
# - 유형끼리 keyword 중복 없음
# - click에 '검색', '접속', '방문' 같은 범용어 넣지 않음
# - participation에 모든 걸 몰아넣되, 구매/가입/설치/실행과 겹치는 단어는 제외
ACTION_TAXONOMY = {
    "action_click": {
        "display_name": "클릭형",
        "keywords": [
            "링크 클릭",
            "버튼 클릭",
            "배너 클릭",
            "광고 클릭",
            "클릭하기",
            "click button",
            "click link",
        ],
        "negative_keywords": [
            "설치", "실행", "구매", "결제", "회원가입",
            "정답", "퀴즈", "설문", "리뷰", "팔로우", "좋아요",
            "구독", "스크린샷", "시청",
        ],
        "seed_docs": [
            "광고 링크나 배너 버튼을 클릭하는 미션",
            "특정 버튼 클릭이 완료 조건인 광고",
        ],
    },

    "action_view": {
        "display_name": "시청형",
        "keywords": [
            "영상 시청",
            "영상시청",
            "동영상 시청",
            "동영상시청",
            "쇼츠 시청",
            "광고 시청",
            "시청 완료",
            "watch video",
            "video view",
        ],
        "negative_keywords": [
            "설치", "실행", "구매", "결제", "회원가입",
            "정답", "퀴즈", "설문", "리뷰", "팔로우", "구독",
            "스크린샷",
        ],
        "seed_docs": [
            "영상을 일정 시간 이상 시청하는 미션",
            "동영상 또는 쇼츠 시청 완료가 조건인 광고",
        ],
    },

    "action_participation": {
        "display_name": "참여형",
        "keywords": [
            "정답 입력",
            "정답입력",
            "퀴즈 맞추기",
            "퀴즈맞추기",
            "설문 참여",
            "이벤트 응모",
            "댓글 작성",
            "리뷰 작성",
            "후기 작성",
            "좋아요 누르기",
            "팔로우하기",
            "채널 구독",
            "게시물 공유",
            "스크린샷 제출",
            "스크린샷 업로드",
            "해시태그 작성",
            "저장하기",
            "찜하기",
            "투표하기",
            "인증하기",
            "미션 참여",
        ],
        "negative_keywords": [
            "앱 설치", "어플 설치", "앱 실행", "어플 실행",
            "회원가입", "계정 생성", "구매 완료", "결제 완료",
        ],
        "seed_docs": [
            "정답 입력 퀴즈 설문 댓글 리뷰 팔로우 좋아요 같은 참여 미션",
            "스크린샷 제출이나 해시태그 작성처럼 사용자가 인증하는 미션",
            "저장하기 찜하기 투표하기처럼 사용자가 직접 행동하는 미션",
        ],
    },

    "action_install": {
        "display_name": "설치형",
        "keywords": [
            "앱 설치",
            "앱설치",
            "어플 설치",
            "어플설치",
            "애플리케이션 설치",
            "다운로드 완료",
            "download complete",
            "install app",
        ],
        "negative_keywords": [
            "앱 실행", "어플 실행", "최초 실행",
            "회원가입", "구매", "결제",
        ],
        "seed_docs": [
            "모바일 앱이나 어플을 설치하는 미션",
            "앱 다운로드 완료가 조건인 광고",
        ],
    },

    "action_run": {
        "display_name": "실행형",
        "keywords": [
            "앱 실행",
            "앱실행",
            "어플 실행",
            "어플실행",
            "최초 실행",
            "튜토리얼 완료",
            "레벨 달성",
            "스테이지 클리어",
            "게임 플레이",
            "run app",
        ],
        "negative_keywords": [
            "앱 설치", "어플 설치", "다운로드",
            "회원가입", "구매", "결제",
        ],
        "seed_docs": [
            "앱 설치 후 실행하거나 최초 실행을 완료하는 미션",
            "게임 플레이 후 레벨 달성 또는 튜토리얼 완료가 조건인 광고",
        ],
    },

    "action_signup": {
        "display_name": "가입형",
        "keywords": [
            "회원가입",
            "회원 가입",
            "가입 완료",
            "가입완료",
            "계정 생성",
            "계정생성",
            "본인인증 완료",
            "사전예약 완료",
            "sign up",
            "create account",
        ],
        "negative_keywords": [
            "구매", "결제", "주문", "앱 설치", "앱 실행",
        ],
        "seed_docs": [
            "서비스 회원가입이나 계정 생성이 완료 조건인 광고",
            "본인인증 또는 사전예약 완료가 조건인 광고",
        ],
    },

    "action_purchase": {
        "display_name": "구매형",
        "keywords": [
            "상품 구매",
            "구매 완료",
            "결제 완료",
            "주문 완료",
            "첫구매",
            "첫 구매",
            "예약 결제",
            "유료 결제",
            "purchase complete",
            "payment complete",
        ],
        "negative_keywords": [
            "회원가입", "계정 생성", "앱 설치", "앱 실행",
            "정답", "퀴즈", "설문",
        ],
        "seed_docs": [
            "상품 구매나 결제 완료가 조건인 광고",
            "주문 완료 또는 첫구매가 조건인 광고",
        ],
    },

    "action_exposure": {
        "display_name": "노출형",
        "keywords": [
            "광고 노출",
            "배너 노출",
            "노출 완료",
            "impression",
            "ad impression",
        ],
        "negative_keywords": [
            "클릭", "시청", "설치", "실행", "구매", "결제",
            "회원가입", "정답", "퀴즈",
        ],
        "seed_docs": [
            "사용자에게 광고나 배너가 노출되는 것 자체가 목표인 광고",
        ],
    },
}


# 운영자가 부여한 ads_type은 텍스트보다 신뢰도 높게 사용
ADS_TYPE_TO_MEDIA = {
    5: "media_facebook",
    6: "media_twitter_x",
    7: "media_instagram",
    10: "media_youtube",
    11: "media_naver",
}

ADS_TYPE_TO_ACTION = {
    1: "action_install",
    2: "action_run",
    3: "action_participation",
    4: "action_click",
    8: "action_exposure",
    12: "action_purchase",
}

## 8. rule-based media 분류

먼저 사람이 만든 규칙으로 media를 분류합니다.

rule-based 접근을 먼저 하는 이유는 설명 가능성이 높기 때문입니다. 예를 들어 naver.com 도메인이 있어서 media_naver로 분류했다면, 왜 그렇게 분류했는지 바로 설명할 수 있습니다.

이 셀은 최종 라벨뿐 아니라 evidence도 저장합니다. evidence는 어떤 단어 또는 도메인 때문에 그 라벨이 선택됐는지를 보여주는 근거입니다.

주의할 점은 여러 후보가 동시에 나올 수 있다는 것입니다. 예를 들어 네이버 블로그 글 안에 유튜브 링크가 있으면 naver와 youtube가 모두 후보가 될 수 있습니다. 이런 경우는 rule_media_multi_candidate 값을 1로 표시해서 검수 대상으로 남깁니다.


In [ ]:
def normalize_for_match(text):
    return normalize_text_basic(text)

# 키워드 보유 여부를 boolean array로 (벡터화)
def build_keyword_mask(text_series_lower, keywords):
    mask = np.zeros(len(text_series_lower), dtype=bool)
    for kw in keywords:
        kw_l = normalize_for_match(kw)
        if not kw_l:
            continue
        mask |= text_series_lower.str.contains(kw_l, regex=False, na=False).values
    return mask

# 도메인 → 매체 후보
def media_from_domains(domains):
    candidates = []
    for domain in domains:
        domain = normalize_domain(domain)
        if domain in DOMAIN_MEDIA_MAP:
            candidates.append((DOMAIN_MEDIA_MAP[domain], f"domain:{domain}"))
            continue
        for known_domain, label in DOMAIN_MEDIA_MAP.items():
            if domain.endswith("." + known_domain) or domain == known_domain:
                candidates.append((label, f"domain:{domain}"))
                break
    return candidates


# ===========================================================
# Cascade 매체 룰
# ===========================================================
# 핵심 변경: 모든 라벨이 동시에 경합하지 않고, 희소 라벨부터 차례로 잡음
# 한 단계에서 라벨이 결정되면 다음 단계로 안 넘어감
# - 1단계 (가장 희소): SNS 직접 단서 (틱톡/페북/트위터/인스타/카카오/유튜브)
# - 2단계: 도메인 매칭
# - 3단계: 앱 패키지 / 앱 명시
# - 4단계: 웹 단서 (회원가입/구매/체험 등)
# - 5단계: 네이버 명시 단서 (직접 키워드)
# - 6단계: 네이버 정황 단서 (퀴즈/플레이스/스크린샷 등)
# - 7단계: ads_type 강제 매핑 (5/6/7/10/11) — 위 단계에서 못 잡힌 행 보강
# - 8단계: 못 잡힌 행은 unknown
# - 참여형(ads_type=3)은 매체 결정에 영향 안 줌 (행동만 결정)
print("매체 룰 cascade 적용 중...")

n = len(df)
text_lower = df["media_doc"].astype(str).str.lower()

# 결과 컨테이너
rule_media = np.array(["media_unknown"] * n, dtype=object)
rule_media_score = np.zeros(n, dtype=float)
rule_media_evidence = [[] for _ in range(n)]
rule_media_stage = np.array([""] * n, dtype=object)
decided = np.zeros(n, dtype=bool)  # 이미 결정된 행

def assign(mask, label, score, stage_label, evidence_label):
    """결정 안 된 행 중에서 mask True인 것에 라벨 부여"""
    target = mask & ~decided
    n_t = target.sum()
    if n_t == 0:
        return 0
    rule_media[target] = label
    rule_media_score[target] = score
    rule_media_stage[target] = stage_label
    for i in np.where(target)[0]:
        rule_media_evidence[i].append(evidence_label)
    decided[target] = True
    return n_t

# 광고 텍스트별 키워드 hit (모든 라벨에 대해 미리 계산)
media_kw_hit = {}
media_neg_hit = {}
for label, info in MEDIA_TAXONOMY.items():
    media_kw_hit[label] = build_keyword_mask(text_lower, info["keywords"])
    media_neg_hit[label] = build_keyword_mask(text_lower, info.get("negative_keywords", []))

# 도메인 점수
strong_dom_hits = df["strong_domains"].map(media_from_domains)
n_app = df["_n_app_packages"].values

# ----- 1단계: SNS 직접 단서 (가장 희소·강한) -----
# 우선순위 순서: tiktok > facebook > twitter_x > instagram > kakao > youtube
SNS_ORDER = ["media_tiktok","media_facebook","media_twitter_x","media_instagram","media_kakao","media_youtube"]
print("\n[stage 1] SNS 직접 키워드")
for label in SNS_ORDER:
    if label not in MEDIA_TAXONOMY:
        continue
    # 키워드 hit & negative 없음
    mask = media_kw_hit[label] & ~media_neg_hit[label]
    n_a = assign(mask, label, 4.0, f"stage1:sns_keyword", f"keyword:{label}")
    print(f"  {label}: {n_a:,}")

# ----- 2단계: 도메인 매칭 (어떤 도메인이든) -----
print("\n[stage 2] 도메인 매칭")
dom_target_label = np.array([""] * n, dtype=object)
for i, hits in enumerate(strong_dom_hits):
    if decided[i]:
        continue
    if hits:
        dom_target_label[i] = hits[0][0]  # 첫 매칭
        # evidence는 도메인 형태 그대로
        rule_media_evidence[i].append(hits[0][1])

for label in MEDIA_LABELS:
    mask = (dom_target_label == label)
    if mask.any():
        n_d = assign(mask, label, 3.5, f"stage2:domain", f"domain_match:{label}")
        if n_d:
            print(f"  {label}: {n_d:,}")

# ----- 3단계: 앱 패키지/앱 명시 → media_app -----
print("\n[stage 3] 앱 패키지/앱 명시")
# 패키지명 있음 → app
app_pkg_mask = (n_app > 0)
n_a = assign(app_pkg_mask, "media_app", 3.0, "stage3:app_package", "app_package")
print(f"  app_package: {n_a:,}")
# 앱 키워드 (taxonomy media_app)
if "media_app" in media_kw_hit:
    mask = media_kw_hit["media_app"] & ~media_neg_hit["media_app"]
    n_a = assign(mask, "media_app", 2.5, "stage3:app_keyword", "keyword:media_app")
    print(f"  app_keyword: {n_a:,}")

# ----- 4단계: 웹 단서 -----
# 웹 직접 키워드 (taxonomy media_web)
print("\n[stage 4] 웹 단서")
if "media_web" in media_kw_hit:
    mask = media_kw_hit["media_web"] & ~media_neg_hit["media_web"]
    n_a = assign(mask, "media_web", 2.5, "stage4:web_keyword", "keyword:media_web")
    print(f"  web_keyword: {n_a:,}")

# ----- 5단계: 네이버 명시 단서 -----
# "네이버", "naver", "스마트스토어", "smartstore", "네이버쇼핑" 등 직접 등장
print("\n[stage 5] 네이버 명시 단서")
if "media_naver" in media_kw_hit:
    # 네이버 명시 키워드 (taxonomy의 keywords)
    naver_explicit_pat = ["네이버","naver","스마트스토어","smartstore","smart store",
                           "n페이","네이버페이","네이버쇼핑","네이버 쇼핑",
                           "네이버 플레이스","네이버 블로그","네이버 카페",
                           "네이버 지도","네이버지도","네이버스마트스토어","n사","n정품"]
    naver_explicit_hit = build_keyword_mask(text_lower, naver_explicit_pat)
    mask = naver_explicit_hit & ~media_neg_hit.get("media_naver", np.zeros(n, dtype=bool))
    n_a = assign(mask, "media_naver", 3.5, "stage5:naver_explicit", "naver_explicit")
    print(f"  naver_explicit: {n_a:,}")

# ----- 6단계: 네이버 정황 단서 (saveway / name) -----
# "퀴즈", "플레이스", "스크린샷", "정답" 등 IVE 운영상 네이버 미션 정황
print("\n[stage 6] 네이버 정황 단서")
naver_circumstantial = ["퀴즈 맞추기","퀴즈맞추기","쇼핑 퀴즈","쇼핑퀴즈",
                          "플레이스","place","검색 후","검색후","상품번호","상품 번호",
                          "스크린샷","해시태그","장소명","거리 입력","명소","스토어찜",
                          "스토어 찜","정답입력","정답 입력","영수증리뷰","영수증 리뷰","찜하기"]
naver_circ_hit = build_keyword_mask(text_lower, naver_circumstantial)
n_a = assign(naver_circ_hit, "media_naver", 2.0, "stage6:naver_circumstantial", "naver_circumstantial")
print(f"  naver_circumstantial: {n_a:,}")

# ----- 7단계: ads_type 강제 매핑 (보강) -----
# 위 단계에서 못 잡힌 ads_type=11 행은 여기서 naver로
print("\n[stage 7] ads_type 강제 매핑 (보강)")
if ADS_TYPE_COL is not None:
    at = df[ADS_TYPE_COL].values
    for type_code, target_label in ADS_TYPE_TO_MEDIA.items():
        if target_label in MEDIA_LABELS:
            mask = (at == type_code)
            n_a = assign(mask, target_label, 5.0, f"stage7:ads_type={type_code}", f"ads_type={type_code}")
            if n_a:
                print(f"  ads_type={type_code} → {target_label}: {n_a:,}")

# ----- 8단계: 미결정 행은 unknown -----
n_unk = (~decided).sum()
print(f"\n[stage 8] unknown으로 남은 행: {n_unk:,}")

# ----- 결과 저장 (rule_media 컬럼 + 부가 정보) -----
rule_media_evidence_str = ["; ".join(ev) for ev in rule_media_evidence]

# multi_candidate: 정황 단서가 여러 라벨에 걸쳐있는지 (간단 표시)
multi_candidate = np.zeros(n, dtype=int)
for i in range(n):
    if not decided[i]:
        continue
    # 다른 라벨도 키워드 hit 있는지
    cnt = 0
    for label in MEDIA_LABELS:
        if media_kw_hit.get(label, np.zeros(n,dtype=bool))[i]:
            cnt += 1
    multi_candidate[i] = 1 if cnt >= 2 else 0

# rule_media_candidates (참고용 — 어떤 라벨에 키워드가 hit됐는지)
rule_media_candidates = []
for i in range(n):
    cand = {}
    for label in MEDIA_LABELS:
        c = 0.0
        if media_kw_hit.get(label, np.zeros(n,dtype=bool))[i]:
            c += 1.0
        if media_neg_hit.get(label, np.zeros(n,dtype=bool))[i]:
            c -= 1.5
        cand[label] = c
    rule_media_candidates.append(json.dumps(cand, ensure_ascii=False))

df["rule_media"] = rule_media
df["rule_media_score"] = rule_media_score
df["rule_media_evidence"] = rule_media_evidence_str
df["rule_media_candidates"] = rule_media_candidates
df["rule_media_multi_candidate"] = multi_candidate
df["rule_media_stage"] = rule_media_stage  # 어느 단계에서 결정됐는지

print("\n=== rule_media 분포 (cascade 적용 후) ===")
display(df["rule_media"].value_counts(dropna=False).to_frame("count"))

print("\n=== 단계별 분포 (어떤 단계에서 결정됐는지) ===")
display(df["rule_media_stage"].value_counts(dropna=False).to_frame("count"))

display(df[[
    "rule_media",
    "rule_media_stage",
    "rule_media_score",
    "rule_media_evidence",
    "media_doc",
]].head(10))


매체 룰 cascade 적용 중...


## 9. rule-based action 분류

action 분류에서는 기본값을 action_click으로 두면 안 됩니다.

왜냐하면 광고 데이터에서 클릭 로그가 많다고 해서 모든 광고가 클릭형 미션인 것은 아니기 때문입니다. 앱 설치형 광고도 클릭 로그가 있고, 구매형 광고도 클릭 로그가 있고, 유튜브 시청형 광고도 클릭 로그가 있을 수 있습니다.

따라서 명확한 근거가 있을 때만 action_click으로 분류하고, 근거가 없으면 action_unknown으로 둡니다.

또한 action은 우선순위가 중요합니다. 예를 들어 앱 설치 후 실행이라는 문구가 있으면 단순 클릭형보다 설치형이나 실행형이 더 중요합니다. 이 노트북에서는 구매, 가입, 설치, 실행, 시청, 참여 같은 구체적인 행동을 클릭보다 우선합니다.


In [ ]:
# Cascade 행동 룰
# 핵심 변경: 희소 행동부터 차례로 잡음. participation은 마지막.
# - 1단계 (희소): purchase / signup / install / run / view / click / exposure
# - 2단계: ads_type 강제 매핑 (1/2/4/8/12)
# - 3단계: ads_type=3 (참여) 강제 매핑
# - 4단계: 미결정은 unknown

ACTION_PRIORITY_CASCADE = [
    "action_purchase",      # 가장 비즈니스 가치 큼
    "action_signup",
    "action_install",
    "action_run",
    "action_view",
    "action_exposure",
    "action_click",
    # action_participation은 마지막 단계로 별도 처리
]

print("행동 룰 cascade 적용 중...")
text_lower_a = df["action_doc"].astype(str).str.lower()

# 키워드 hit 미리 계산
action_kw_hit = {}
action_neg_hit = {}
for label, info in ACTION_TAXONOMY.items():
    action_kw_hit[label] = build_keyword_mask(text_lower_a, info["keywords"])
    action_neg_hit[label] = build_keyword_mask(text_lower_a, info.get("negative_keywords", []))

# 결과 컨테이너
rule_action = np.array(["action_unknown"] * n, dtype=object)
rule_action_score = np.zeros(n, dtype=float)
rule_action_evidence = [[] for _ in range(n)]
rule_action_stage = np.array([""] * n, dtype=object)
decided_a = np.zeros(n, dtype=bool)

def assign_a(mask, label, score, stage_label, evidence_label):
    target = mask & ~decided_a
    n_t = target.sum()
    if n_t == 0:
        return 0
    rule_action[target] = label
    rule_action_score[target] = score
    rule_action_stage[target] = stage_label
    for i in np.where(target)[0]:
        rule_action_evidence[i].append(evidence_label)
    decided_a[target] = True
    return n_t

# ----- 1단계: 희소 행동 키워드 (희소 → 빈출 순서, participation 제외) -----
print("\n[stage 1] 희소 행동 키워드 (purchase, signup, install, run, view, exposure, click)")
for label in ACTION_PRIORITY_CASCADE:
    if label not in action_kw_hit:
        continue
    mask = action_kw_hit[label] & ~action_neg_hit[label]
    n_a = assign_a(mask, label, 2.5, f"stage1:{label}", f"keyword:{label}")
    if n_a:
        print(f"  {label}: {n_a:,}")

# ----- 2단계: ads_type 강제 매핑 (1/2/4/8/12 — 참여형 제외) -----
print("\n[stage 2] ads_type 강제 매핑 (참여형 제외)")
if ADS_TYPE_COL is not None:
    at = df[ADS_TYPE_COL].values
    # 참여형(3)은 마지막 단계로 미룸
    for type_code, target_label in ADS_TYPE_TO_ACTION.items():
        if type_code == 3:  # 참여형은 stage 3에서 처리
            continue
        if target_label in ACTION_LABELS:
            mask = (at == type_code)
            n_a = assign_a(mask, target_label, 5.0, f"stage2:ads_type={type_code}", f"ads_type={type_code}")
            if n_a:
                print(f"  ads_type={type_code} → {target_label}: {n_a:,}")

# 패키지명이 있으면 install 보강 (이미 1단계에서 잡혔다면 건드리지 않음)
app_mask_a = df["_n_app_packages"].values > 0
n_a = assign_a(app_mask_a, "action_install", 1.5, "stage2:app_package", "app_package_hint")
if n_a:
    print(f"  app_package_hint → install: {n_a:,}")

# ----- 3단계: ads_type=3 → participation -----
print("\n[stage 3] ads_type=3 → action_participation")
if ADS_TYPE_COL is not None:
    at = df[ADS_TYPE_COL].values
    mask = (at == 3)
    n_a = assign_a(mask, "action_participation", 4.0, "stage3:ads_type=3", "ads_type=3")
    print(f"  ads_type=3 → participation: {n_a:,}")

# 참여형 키워드 매칭 (마지막 보강)
if "action_participation" in action_kw_hit:
    mask = action_kw_hit["action_participation"] & ~action_neg_hit["action_participation"]
    n_a = assign_a(mask, "action_participation", 1.5, "stage3:participation_keyword", "keyword:action_participation")
    if n_a:
        print(f"  participation_keyword: {n_a:,}")

# ----- 4단계: 미결정 → unknown -----
n_unk_a = (~decided_a).sum()
print(f"\n[stage 4] unknown으로 남은 행: {n_unk_a:,}")

# ----- 결과 저장 -----
rule_action_evidence_str = ["; ".join(ev) for ev in rule_action_evidence]

# multi_candidate
multi_candidate_a = np.zeros(n, dtype=int)
for i in range(n):
    if not decided_a[i]:
        continue
    cnt = 0
    for label in ACTION_LABELS:
        if action_kw_hit.get(label, np.zeros(n, dtype=bool))[i]:
            cnt += 1
    multi_candidate_a[i] = 1 if cnt >= 2 else 0

rule_action_candidates = []
for i in range(n):
    cand = {}
    for label in ACTION_LABELS:
        c = 0.0
        if action_kw_hit.get(label, np.zeros(n, dtype=bool))[i]:
            c += 1.0
        if action_neg_hit.get(label, np.zeros(n, dtype=bool))[i]:
            c -= 1.5
        cand[label] = c
    rule_action_candidates.append(json.dumps(cand, ensure_ascii=False))

df["rule_action"] = rule_action
df["rule_action_score"] = rule_action_score
df["rule_action_evidence"] = rule_action_evidence_str
df["rule_action_candidates"] = rule_action_candidates
df["rule_action_multi_candidate"] = multi_candidate_a
df["rule_action_stage"] = rule_action_stage

print("\n=== rule_action 분포 (cascade 적용 후) ===")
display(df["rule_action"].value_counts(dropna=False).to_frame("count"))

print("\n=== 단계별 분포 ===")
display(df["rule_action_stage"].value_counts(dropna=False).to_frame("count"))

display(df[[
    "rule_action",
    "rule_action_stage",
    "rule_action_score",
    "rule_action_evidence",
    "action_doc",
]].head(10))


## 10. rule 결과 해석하기

여기까지는 머신러닝이 아니라 사람이 만든 규칙입니다.

이 단계에서 확인해야 할 것은 두 가지입니다.

첫째, media_unknown이 너무 많다면 media taxonomy가 부족하거나 텍스트에서 중요한 단서를 놓치고 있는 것입니다. 특히 URL과 도메인이 제대로 추출됐는지 확인해야 합니다.

둘째, action_click이 너무 많다면 클릭을 너무 쉽게 분류하고 있는지 확인해야 합니다. 이 노트북에서는 action_click을 기본값으로 쓰지 않았기 때문에, action_click이 많다면 실제로 클릭/방문 관련 단어가 많이 들어있다는 뜻일 가능성이 큽니다. 그래도 evidence를 꼭 확인해야 합니다.


In [ ]:
print("rule_media 분포")
display(df["rule_media"].value_counts(dropna=False).to_frame("count"))

print("rule_action 분포")
display(df["rule_action"].value_counts(dropna=False).to_frame("count"))

# unknown 샘플을 확인합니다.
# unknown이 진짜 정보 부족인지, 아니면 규칙이 부족해서 놓친 것인지 사람이 봐야 합니다.
unknown_preview_cols = [
    "rule_media", "rule_action",
    "rule_media_evidence", "rule_action_evidence",
    "strong_domains", "weak_domains", "app_packages_detected",
    "classification_doc",
]
unknown_preview_cols = [c for c in unknown_preview_cols if c in df.columns]

unknown_preview = df[
    (df["rule_media"] == "media_unknown") | (df["rule_action"] == "action_unknown")
][unknown_preview_cols].head(20)

display(unknown_preview)


## 11. BERTopic을 위한 embedding 생성

이제 머신러닝을 사용합니다.

BERTopic은 텍스트를 바로 처리하지 않고, 먼저 문장을 숫자 벡터로 바꿉니다. 이 과정을 embedding이라고 합니다.

이 노트북에서는 sentence-transformers 모델을 사용합니다.

초보자에게 중요한 포인트는 모델을 바꾼다고 무조건 성능이 좋아지는 것이 아니라는 점입니다. 현재 문제는 입력 텍스트와 규칙 설계가 더 중요합니다. 그래도 한국어 광고 문구에서는 한국어에 강한 embedding 모델을 쓰면 도움이 될 수 있습니다.

기본 모델은 속도와 안정성을 위해 multilingual 모델로 둡니다. GPU가 있거나 시간이 충분하면 Korean 모델로 바꿔서 비교해보세요.


In [ ]:
from sentence_transformers import SentenceTransformer

# 기본 모델입니다.
# 다국어를 지원하고 속도가 비교적 빠릅니다.
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# 한국어 중심 모델을 시험해보고 싶으면 아래 중 하나로 바꿔볼 수 있습니다.
# EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"
# EMBEDDING_MODEL_NAME = "BM-K/KoSimCSE-roberta-multitask"
# EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# BERTopic에 넣을 문서입니다.
# 빈 문서는 학습에 도움이 되지 않으므로 placeholder를 넣습니다.
docs = df["classification_doc"].fillna("").astype(str).tolist()
docs = [d if d.strip() else "empty document" for d in docs]

print("문서 수:", len(docs))
print("첫 번째 문서 예시:")
print(docs[0][:1000])

# embedding을 생성합니다.
# normalize_embeddings=True는 cosine similarity 계산을 안정적으로 해줍니다.
embeddings = embedding_model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("embedding shape:", embeddings.shape)


## 12. BERTopic 학습

BERTopic은 비슷한 광고 문서를 topic으로 묶습니다.

여기서 topic 번호는 라벨이 아닙니다. 예를 들어 topic 3이 media_youtube라는 뜻이 아닙니다. topic 3은 그냥 비슷한 광고 묶음의 번호입니다.

BERTopic 결과에서 -1은 outlier를 의미합니다. 어느 topic에도 안정적으로 들어가지 못한 문서입니다.

min_topic_size는 하나의 topic이 되기 위한 최소 문서 수입니다. 너무 작게 잡으면 topic이 너무 많이 생기고, 너무 크게 잡으면 다양한 광고가 한 topic에 섞일 수 있습니다.


In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# UMAP은 고차원 embedding을 낮은 차원으로 줄여주는 역할입니다.
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
)

# HDBSCAN은 밀도 기반 군집화 알고리즘입니다.
hdbscan_model = HDBSCAN(
    min_cluster_size=25,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

# CountVectorizer는 topic을 설명하는 대표 단어를 뽑을 때 사용합니다.
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=2,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True,
)

topics, topic_probs = topic_model.fit_transform(docs, embeddings)
df["bertopic_topic"] = topics

topic_info = topic_model.get_topic_info()
display(topic_info.head(20))

print("topic 분포 상위 20개:")
display(df["bertopic_topic"].value_counts().head(20).to_frame("count"))


## 13. topic 대표 문서 확인하기

BERTopic 결과는 반드시 사람이 확인해야 합니다.

아래 함수는 특정 topic에 속한 광고 문서 예시와 rule 분류 결과를 보여줍니다.

학생들이 자주 하는 실수는 topic 단어만 보고 라벨을 확정하는 것입니다. topic 단어는 참고용일 뿐이고, 실제 광고 문구 예시를 함께 봐야 합니다.


In [ ]:
def show_topic_examples(topic_id, n=8):
    # 특정 topic의 예시 문서를 보여줍니다.
    sub = df[df["bertopic_topic"] == topic_id].copy()

    print("topic:", topic_id)
    print("row count:", len(sub))

    if topic_id != -1:
        try:
            print("top words:", topic_model.get_topic(topic_id)[:10])
        except Exception:
            pass

    cols = [
        "rule_media",
        "rule_action",
        "rule_media_evidence",
        "rule_action_evidence",
        "strong_domains",
        "weak_domains",
        "app_packages_detected",
        "classification_doc",
    ]
    cols = [c for c in cols if c in sub.columns]

    display(sub[cols].head(n))

# 가장 큰 topic 몇 개를 자동으로 확인합니다.
top_topics = df["bertopic_topic"].value_counts().head(5).index.tolist()

for t in top_topics:
    show_topic_examples(t, n=5)


## 14. seed 문장으로 topic 후보 라벨 만들기

이제 각 topic이 어떤 media/action에 가까운지 추정합니다.

방법은 간단합니다.

먼저 각 라벨을 설명하는 seed 문장을 준비합니다. 예를 들어 media_youtube의 seed 문장은 유튜브 영상을 시청하고 구독하는 광고 같은 문장입니다.

그 다음 topic에 속한 문서들의 embedding 평균과 seed 문장의 embedding을 비교합니다.

가장 비슷한 seed 라벨을 topic 후보 라벨로 둡니다.

하지만 이것은 정답이 아닙니다. topic 후보일 뿐입니다. 그래서 score와 margin을 같이 봅니다.

score는 가장 비슷한 라벨과의 유사도입니다.

margin은 1등 라벨과 2등 라벨의 점수 차이입니다. margin이 작으면 헷갈린다는 뜻입니다.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def build_seed_df(taxonomy, label_col_name):
    # taxonomy 안의 seed_docs를 DataFrame으로 바꿉니다.
    rows = []
    for label, info in taxonomy.items():
        for seed_doc in info.get("seed_docs", []):
            rows.append({
                label_col_name: label,
                "display_name": info.get("display_name", label),
                "seed_doc": seed_doc,
            })
    return pd.DataFrame(rows)

def score_topics_with_seeds(taxonomy, label_col_name, prefix):
    # 각 BERTopic topic이 어떤 라벨 seed와 가장 가까운지 계산합니다.
    seed_df = build_seed_df(taxonomy, label_col_name)

    seed_embeddings = embedding_model.encode(
        seed_df["seed_doc"].tolist(),
        normalize_embeddings=True,
    )

    topic_info = topic_model.get_topic_info()
    valid_topics = [int(t) for t in topic_info["Topic"].tolist() if int(t) != -1]

    rows = []

    for topic_id in valid_topics:
        idx = np.where(df["bertopic_topic"].values == topic_id)[0]
        if len(idx) == 0:
            continue

        # topic에 속한 문서들의 embedding 평균을 topic embedding처럼 사용합니다.
        topic_embedding = embeddings[idx].mean(axis=0, keepdims=True)

        # topic embedding과 seed embedding 사이의 cosine similarity를 계산합니다.
        similarities = cosine_similarity(topic_embedding, seed_embeddings)[0]

        temp = seed_df.copy()
        temp["similarity"] = similarities

        # 라벨별로 seed 문장이 여러 개 있을 수 있으므로, 가장 높은 점수를 라벨 점수로 사용합니다.
        label_scores = (
            temp.groupby(label_col_name)["similarity"]
            .max()
            .sort_values(ascending=False)
        )

        best_label = label_scores.index[0]
        best_score = float(label_scores.iloc[0])
        second_score = float(label_scores.iloc[1]) if len(label_scores) > 1 else np.nan
        margin = best_score - second_score if not np.isnan(second_score) else np.nan

        try:
            top_words = topic_model.get_topic(topic_id)
            top_words_text = ", ".join([w for w, _ in top_words[:10]])
        except Exception:
            top_words_text = ""

        topic_docs = df.loc[idx, "classification_doc"].head(5).tolist()

        row = {
            "topic": int(topic_id),
            "topic_size": int(len(idx)),
            f"{prefix}_best_label": best_label,
            f"{prefix}_best_score": best_score,
            f"{prefix}_second_score": second_score,
            f"{prefix}_score_margin": margin,
            "top_words": top_words_text,
            "representative_docs": "\n---\n".join(topic_docs),
        }

        for label, score in label_scores.items():
            row[f"{label}_seed_score"] = float(score)

        rows.append(row)

    return pd.DataFrame(rows)

topic_media_score_df = score_topics_with_seeds(MEDIA_TAXONOMY, "media", "media")
topic_action_score_df = score_topics_with_seeds(ACTION_TAXONOMY, "action", "action")

print("topic media score 예시:")
display(topic_media_score_df.head(10))

print("topic action score 예시:")
display(topic_action_score_df.head(10))


## 15. topic 내부 rule 비율 계산하기

seed similarity만 보면 오탐이 생길 수 있습니다.

그래서 각 topic 안에서 rule 결과가 얼마나 한 라벨로 모이는지도 함께 봅니다.

예를 들어 topic 10 안의 광고 80%가 rule_media에서 media_naver로 잡혔다면, 이 topic은 media_naver일 가능성이 높습니다.

반대로 topic 안에서 rule 라벨이 섞여 있으면, 그 topic을 하나의 라벨로 자동 채우는 것은 위험합니다.


In [ ]:
def topic_rule_share(label_col, labels, unknown_label, prefix):
    # topic별로 rule 라벨 비율을 계산합니다.
    rows = []
    valid_topics = sorted([t for t in df["bertopic_topic"].unique() if t != -1])

    for topic_id in valid_topics:
        sub = df[df["bertopic_topic"] == topic_id]
        row = {"topic": int(topic_id)}

        for label in labels:
            row[f"{label}_rule_share"] = float((sub[label_col] == label).mean()) if len(sub) else 0.0

        row[f"{prefix}_unknown_rule_share"] = float((sub[label_col] == unknown_label).mean()) if len(sub) else 0.0
        rows.append(row)

    return pd.DataFrame(rows)

topic_media_rule_share_df = topic_rule_share("rule_media", MEDIA_LABELS, "media_unknown", "media")
topic_action_rule_share_df = topic_rule_share("rule_action", ACTION_LABELS, "action_unknown", "action")

topic_media_review = topic_media_score_df.merge(topic_media_rule_share_df, on="topic", how="left")
topic_action_review = topic_action_score_df.merge(topic_action_rule_share_df, on="topic", how="left")

display(topic_media_review.head(10))
display(topic_action_review.head(10))


## 16. topic 후보 라벨의 신뢰도 판정

이제 topic 후보를 자동으로 사용할지 말지를 결정합니다.

이 노트북은 보수적으로 설계되어 있습니다. 즉, 확실한 경우에만 topic으로 unknown을 채우고, 애매하면 unknown으로 남깁니다.

threshold 값의 의미는 다음과 같습니다.

score_th는 seed 문장과 topic의 유사도가 이 값 이상이어야 한다는 기준입니다.

margin_th는 1등 라벨과 2등 라벨의 차이가 이 값 이상이어야 한다는 기준입니다.

rule_share_th는 topic 안에서 같은 rule 라벨이 이 비율 이상이어야 한다는 기준입니다.

초보자에게 중요한 점은 threshold를 낮추면 unknown은 줄지만 오분류가 늘 수 있다는 것입니다. 그래서 threshold는 검수셋을 만든 뒤 조정해야 합니다.


In [ ]:
# 토픽 후보를 사용할 기준입니다.
# 이전 점검 결과 best_score 평균이 0.56, margin 99.8%가 < 0.1이었습니다.
# 너무 낮은 임계값은 잘못된 토픽 라벨을 high_confidence로 표기하게 만듭니다.
# 보수적으로 올렸습니다.
MEDIA_TOPIC_SCORE_TH = 0.50      # 0.38 -> 0.50
MEDIA_TOPIC_MARGIN_TH = 0.05     # 0.03 -> 0.05
MEDIA_TOPIC_RULE_SHARE_TH = 0.50 # 0.35 -> 0.50

ACTION_TOPIC_SCORE_TH = 0.50
ACTION_TOPIC_MARGIN_TH = 0.05
ACTION_TOPIC_RULE_SHARE_TH = 0.50

# True면 rule에서 unknown인 행을 topic 후보로 일부 채웁니다.
# 정답셋이 검증되기 전에는 False로 두는 것을 권장합니다.
USE_TOPIC_TO_FILL_UNKNOWN = True

def decide_topic_label(row, prefix, score_th, margin_th, rule_share_th, unknown_label):
    best_label = row.get(f"{prefix}_best_label", unknown_label)
    best_score = row.get(f"{prefix}_best_score", 0.0)
    margin = row.get(f"{prefix}_score_margin", 0.0)
    rule_share = row.get(f"{best_label}_rule_share", 0.0)

    score_condition = (best_score >= score_th) and (margin >= margin_th)
    rule_condition = rule_share >= rule_share_th

    if score_condition or rule_condition:
        return best_label
    return unknown_label

topic_media_review["topic_media_candidate"] = topic_media_review.apply(
    lambda row: decide_topic_label(
        row, "media", MEDIA_TOPIC_SCORE_TH, MEDIA_TOPIC_MARGIN_TH,
        MEDIA_TOPIC_RULE_SHARE_TH, "media_unknown",
    ), axis=1,
)
topic_action_review["topic_action_candidate"] = topic_action_review.apply(
    lambda row: decide_topic_label(
        row, "action", ACTION_TOPIC_SCORE_TH, ACTION_TOPIC_MARGIN_TH,
        ACTION_TOPIC_RULE_SHARE_TH, "action_unknown",
    ), axis=1,
)

topic_media_review["media_needs_review"] = (
    (topic_media_review["topic_media_candidate"] == "media_unknown")
    | (topic_media_review["media_best_score"] < MEDIA_TOPIC_SCORE_TH)
    | (topic_media_review["media_score_margin"] < MEDIA_TOPIC_MARGIN_TH)
).astype(int)

topic_action_review["action_needs_review"] = (
    (topic_action_review["topic_action_candidate"] == "action_unknown")
    | (topic_action_review["action_best_score"] < ACTION_TOPIC_SCORE_TH)
    | (topic_action_review["action_score_margin"] < ACTION_TOPIC_MARGIN_TH)
).astype(int)

print("topic_media_candidate 분포:")
display(topic_media_review["topic_media_candidate"].value_counts(dropna=False).to_frame("count"))
print("topic_action_candidate 분포:")
display(topic_action_review["topic_action_candidate"].value_counts(dropna=False).to_frame("count"))


## 17. 최종 media/action 라벨 만들기

최종 라벨을 정하는 우선순위는 다음과 같습니다.

첫째, rule에서 확실히 잡힌 라벨을 우선합니다. rule은 evidence가 명확하기 때문입니다.

둘째, rule이 unknown인 경우에만 topic 후보를 사용합니다. 즉, topic이 rule을 덮어쓰지 않게 합니다.

셋째, 둘 다 근거가 없으면 unknown으로 남깁니다.

이 방식은 coverage를 무리하게 늘리기보다 precision을 지키는 방향입니다. 실무에서는 틀린 라벨을 대량으로 붙이는 것이 더 큰 문제가 될 수 있습니다.


In [ ]:
topic_to_media = dict(zip(topic_media_review["topic"], topic_media_review["topic_media_candidate"]))
topic_to_action = dict(zip(topic_action_review["topic"], topic_action_review["topic_action_candidate"]))

df["topic_media_candidate"] = df["bertopic_topic"].map(topic_to_media).fillna("media_unknown")
df["topic_action_candidate"] = df["bertopic_topic"].map(topic_to_action).fillna("action_unknown")

# 벡터화된 final 선택
def choose_final_vectorized(rule_col, topic_col, unknown_label, use_topic):
    rule = df[rule_col].values
    topic = df[topic_col].values
    final = rule.copy()
    source = np.where(rule != unknown_label, "rule", "unknown")
    if use_topic:
        fill_mask = (rule == unknown_label) & (topic != unknown_label)
        final[fill_mask] = topic[fill_mask]
        source[fill_mask] = "topic_high_confidence"
    return final, source

final_media, media_source = choose_final_vectorized(
    "rule_media", "topic_media_candidate", "media_unknown", USE_TOPIC_TO_FILL_UNKNOWN
)
final_action, action_source = choose_final_vectorized(
    "rule_action", "topic_action_candidate", "action_unknown", USE_TOPIC_TO_FILL_UNKNOWN
)
df["final_media"] = final_media
df["media_source"] = media_source
df["final_action"] = final_action
df["action_source"] = action_source

# ★ save_way 일관성 후처리 (수정본)
if "ads_save_way" in df.columns:
    print("save_way 일관성 후처리 중...")
    sw = df["ads_save_way"].fillna("").astype(str).str.strip()
    valid_sw_mask = sw != ""

    # 다수결 기준을 rule로만 제한
    rule_only_mask = (
        valid_sw_mask
        & (df["final_media"] != "media_unknown")
        & (df["media_source"] == "rule")
    )
    sw_known = df.loc[rule_only_mask, :]

    if len(sw_known) > 0:
        sw_majority_media = sw_known.groupby(
            sw[rule_only_mask]
        )["final_media"].agg(
            lambda s: s.value_counts().index[0] if len(s) else "media_unknown"
        )
        sw_top_share = sw_known.groupby(
            sw[rule_only_mask]
        )["final_media"].agg(
            lambda s: s.value_counts(normalize=True).iloc[0] if len(s) else 0.0
        )
        sw_count = sw_known.groupby(sw[rule_only_mask])["final_media"].count()

        trusted_sw = sw_top_share[
            (sw_top_share >= 0.90) &
            (sw_count >= 5)
        ].index
        sw_to_media = sw_majority_media[trusted_sw].to_dict()

        print(f"  신뢰 save_way 수: {len(trusted_sw):,}개")

        unknown_mask = (df["final_media"] == "media_unknown") & valid_sw_mask
        idxs = df.index[unknown_mask]
        filled = 0
        for idx in idxs:
            sw_val = sw.iloc[df.index.get_loc(idx)]
            if sw_val in sw_to_media:
                df.at[idx, "final_media"] = sw_to_media[sw_val]
                df.at[idx, "media_source"] = "save_way_majority"
                filled += 1
        print(f"  save_way 다수결로 채운 행: {filled:,}")

# one-hot flag 생성
for label in MEDIA_LABELS + ["media_unknown"]:
    df[label] = (df["final_media"] == label).astype(int)
for label in ACTION_LABELS + ["action_unknown"]:
    df[label] = (df["final_action"] == label).astype(int)

print("final_media 분포:")
display(df["final_media"].value_counts(dropna=False).to_frame("count"))
print("final_action 분포:")
display(df["final_action"].value_counts(dropna=False).to_frame("count"))
print("media_source 분포:")
display(df["media_source"].value_counts(dropna=False).to_frame("count"))
print("action_source 분포:")
display(df["action_source"].value_counts(dropna=False).to_frame("count"))

## 18. 검수용 샘플 만들기

정확도를 계산하려면 정답이 필요합니다. 그런데 현재 데이터에는 true_media, true_action 같은 정답 컬럼이 없습니다.

그래서 사람이 직접 라벨링할 검수 샘플을 만듭니다.

샘플은 무작위로만 뽑으면 안 됩니다. unknown, topic으로 채운 행, 후보가 여러 개인 행처럼 위험한 케이스를 일부러 많이 뽑아야 합니다.

검수자는 true_media와 true_action 컬럼에 직접 정답을 입력하면 됩니다.


In [ ]:
def sample_safely(frame, n, random_state=RANDOM_STATE):
    # 데이터가 n개보다 적어도 에러가 나지 않도록 안전하게 샘플링합니다.
    if len(frame) == 0:
        return frame
    return frame.sample(min(n, len(frame)), random_state=random_state)

unknown_sample = sample_safely(
    df[(df["final_media"] == "media_unknown") | (df["final_action"] == "action_unknown")],
    120,
)

topic_sample = sample_safely(
    df[(df["media_source"] == "topic_high_confidence") | (df["action_source"] == "topic_high_confidence")],
    80,
)

multi_sample = sample_safely(
    df[(df["rule_media_multi_candidate"] == 1) | (df["rule_action_multi_candidate"] == 1)],
    60,
)

normal_sample = sample_safely(
    df[(df["media_source"] == "rule") & (df["action_source"] == "rule")],
    80,
)

validation_sample = pd.concat(
    [unknown_sample, topic_sample, multi_sample, normal_sample],
    axis=0,
)

# drop_duplicates() 대신 index 기준으로 중복 제거
validation_sample = validation_sample[~validation_sample.index.duplicated(keep="first")]
# 사람이 직접 채울 정답 컬럼입니다.
validation_sample["true_media"] = ""
validation_sample["true_action"] = ""
validation_sample["review_note"] = ""

validation_cols = [
    "true_media",
    "true_action",
    "review_note",
    "final_media",
    "final_action",
    "media_source",
    "action_source",
    "rule_media",
    "rule_action",
    "topic_media_candidate",
    "topic_action_candidate",
    "rule_media_score",
    "rule_action_score",
    "rule_media_evidence",
    "rule_action_evidence",
    "rule_media_candidates",
    "rule_action_candidates",
    "bertopic_topic",
    "strong_text",
    "weak_text",
    "strong_domains",
    "weak_domains",
    "app_packages_detected",
    "classification_doc",
]

validation_cols = [c for c in validation_cols if c in validation_sample.columns]

print("검수 샘플 수:", len(validation_sample))
display(validation_sample[validation_cols].head(20))


## 19. 검수 완료 파일로 성능 평가하기

이 셀은 사람이 true_media와 true_action을 채운 뒤에 사용합니다.

과정은 다음과 같습니다.

먼저 이 노트북이 validation_labeling_sample.xlsx 파일을 저장합니다.

그 파일을 열어서 true_media와 true_action을 직접 채웁니다.

채운 파일을 validation_labeling_sample_filled.xlsx라는 이름으로 같은 폴더에 저장합니다.

그 다음 아래 셀을 다시 실행하면 classification_report가 나옵니다.

classification_report에서 볼 값은 precision, recall, f1-score입니다.

precision은 모델이 이 라벨이라고 예측한 것 중 실제로 맞은 비율입니다.

recall은 실제 이 라벨인 것 중 모델이 얼마나 잘 찾아냈는지입니다.

f1-score는 precision과 recall을 함께 반영한 점수입니다.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

VALIDATED_FILE = OUT_DIR / "validation_labeling_sample_filled.xlsx"

def evaluate_if_validated(path=VALIDATED_FILE):
    # 사람이 채운 검수 파일이 있으면 성능을 계산합니다.
    # 파일이 없으면 안내 메시지만 출력합니다.
    if not path.exists():
        print("아직 검수 완료 파일이 없습니다.")
        print("검수 파일을 채운 뒤 아래 경로로 저장하면 평가가 실행됩니다.")
        print(path)
        return None

    val = pd.read_excel(path, engine="openpyxl")

    results = {}

    if "true_media" in val.columns and val["true_media"].replace("", np.nan).notna().sum() > 0:
        media_eval = val[val["true_media"].replace("", np.nan).notna()].copy()
        print("media 평가 대상 수:", len(media_eval))
        print(classification_report(media_eval["true_media"], media_eval["final_media"], zero_division=0))

        media_labels = sorted(set(media_eval["true_media"].dropna()) | set(media_eval["final_media"].dropna()))
        results["media_confusion"] = pd.DataFrame(
            confusion_matrix(media_eval["true_media"], media_eval["final_media"], labels=media_labels),
            index=media_labels,
            columns=media_labels,
        )

        print("media confusion matrix:")
        display(results["media_confusion"])

    if "true_action" in val.columns and val["true_action"].replace("", np.nan).notna().sum() > 0:
        action_eval = val[val["true_action"].replace("", np.nan).notna()].copy()
        print("action 평가 대상 수:", len(action_eval))
        print(classification_report(action_eval["true_action"], action_eval["final_action"], zero_division=0))

        action_labels = sorted(set(action_eval["true_action"].dropna()) | set(action_eval["final_action"].dropna()))
        results["action_confusion"] = pd.DataFrame(
            confusion_matrix(action_eval["true_action"], action_eval["final_action"], labels=action_labels),
            index=action_labels,
            columns=action_labels,
        )

        print("action confusion matrix:")
        display(results["action_confusion"])

    return results

eval_results = evaluate_if_validated()


## 20. threshold 튜닝하기

검수셋이 만들어진 뒤에는 threshold를 조정할 수 있습니다.

주의할 점은 unknown을 무작정 줄이는 방향으로 튜닝하면 안 된다는 것입니다. unknown이 줄어도 오답이 늘면 실무적으로 더 나쁩니다.

이 셀은 여러 threshold 조합을 시험해서 accuracy와 unknown rate를 비교합니다.

검수 파일이 없으면 실행해도 아무 결과가 나오지 않습니다.


In [ ]:
def simulate_topic_fill(val, media_score_th, media_margin_th, action_score_th, action_margin_th):
    # threshold를 바꿨을 때 최종 라벨이 어떻게 달라지는지 시뮬레이션합니다.
    temp = val.copy()

    media_topic_meta = topic_media_review[[
        "topic", "media_best_label", "media_best_score", "media_score_margin"
    ]].rename(columns={
        "media_best_label": "sim_media_best_label",
        "media_best_score": "sim_media_best_score",
        "media_score_margin": "sim_media_score_margin",
    })

    action_topic_meta = topic_action_review[[
        "topic", "action_best_label", "action_best_score", "action_score_margin"
    ]].rename(columns={
        "action_best_label": "sim_action_best_label",
        "action_best_score": "sim_action_best_score",
        "action_score_margin": "sim_action_score_margin",
    })

    temp = temp.merge(media_topic_meta, left_on="bertopic_topic", right_on="topic", how="left")
    temp = temp.merge(action_topic_meta, left_on="bertopic_topic", right_on="topic", how="left", suffixes=("", "_action_meta"))

    temp["sim_final_media"] = temp["rule_media"]
    media_fill_mask = (
        (temp["rule_media"] == "media_unknown")
        & (temp["sim_media_best_score"] >= media_score_th)
        & (temp["sim_media_score_margin"] >= media_margin_th)
    )
    temp.loc[media_fill_mask, "sim_final_media"] = temp.loc[media_fill_mask, "sim_media_best_label"]

    temp["sim_final_action"] = temp["rule_action"]
    action_fill_mask = (
        (temp["rule_action"] == "action_unknown")
        & (temp["sim_action_best_score"] >= action_score_th)
        & (temp["sim_action_score_margin"] >= action_margin_th)
    )
    temp.loc[action_fill_mask, "sim_final_action"] = temp.loc[action_fill_mask, "sim_action_best_label"]

    return temp

def threshold_grid_search(validated_path=VALIDATED_FILE):
    # 여러 threshold 조합을 비교합니다.
    if not validated_path.exists():
        print("검수 완료 파일이 없어서 threshold grid search를 건너뜁니다.")
        return pd.DataFrame()

    val = pd.read_excel(validated_path, engine="openpyxl")

    grid_rows = []
    media_score_grid = [0.30, 0.35, 0.38, 0.40, 0.45]
    margin_grid = [0.01, 0.02, 0.03, 0.05]
    action_score_grid = [0.30, 0.35, 0.38, 0.40, 0.45]

    for ms in media_score_grid:
        for mm in margin_grid:
            for acs in action_score_grid:
                for am in margin_grid:
                    sim = simulate_topic_fill(val, ms, mm, acs, am)

                    row = {
                        "media_score_th": ms,
                        "media_margin_th": mm,
                        "action_score_th": acs,
                        "action_margin_th": am,
                        "media_unknown_rate": float((sim["sim_final_media"] == "media_unknown").mean()),
                        "action_unknown_rate": float((sim["sim_final_action"] == "action_unknown").mean()),
                    }

                    if "true_media" in sim.columns and sim["true_media"].replace("", np.nan).notna().sum() > 0:
                        media_eval = sim[sim["true_media"].replace("", np.nan).notna()].copy()
                        row["media_accuracy"] = float((media_eval["true_media"] == media_eval["sim_final_media"]).mean())

                    if "true_action" in sim.columns and sim["true_action"].replace("", np.nan).notna().sum() > 0:
                        action_eval = sim[sim["true_action"].replace("", np.nan).notna()].copy()
                        row["action_accuracy"] = float((action_eval["true_action"] == action_eval["sim_final_action"]).mean())

                    grid_rows.append(row)

    grid_df = pd.DataFrame(grid_rows).sort_values(
        ["media_accuracy", "action_accuracy", "media_unknown_rate", "action_unknown_rate"],
        ascending=[False, False, True, True],
        na_position="last",
    )

    return grid_df

threshold_grid_df = threshold_grid_search()

if len(threshold_grid_df):
    display(threshold_grid_df.head(20))


## 21. 검수용 테이블 만들기

최종 결과를 저장하기 전에 검수 우선순위가 높은 행을 따로 모읍니다.

low_confidence_review는 unknown이 있거나 topic으로 채웠거나 후보가 여러 개인 행입니다. 이 테이블을 먼저 보면 성능 개선 포인트를 찾기 쉽습니다.

weak_url_review는 weak domain만 있는 행입니다. 이 경우 공식 SNS 링크 같은 보조 링크 때문에 media가 잘못 잡힐 수 있어 따로 확인합니다.


In [ ]:
MAX_REVIEW_ROWS = 5000

base_cols = []
for c in [
    "ads_idx",
    "ads_id",
    "id",
    "idx",
    "ads_code",
    "ads_type",
    "ads_category",
    "ads_name",
    "ads_title",
    "title",
    "name",
    "ads_search",
    "ads_save_way",
    "ads_package",
]:
    if c in df.columns and c not in base_cols:
        base_cols.append(c)

result_cols = (
    base_cols
    + [
        "final_media",
        "final_action",
        "media_source",
        "action_source",
        "rule_media",
        "rule_action",
        "topic_media_candidate",
        "topic_action_candidate",
        "rule_media_score",
        "rule_action_score",
        "rule_media_evidence",
        "rule_action_evidence",
        "rule_media_candidates",
        "rule_action_candidates",
        "rule_media_multi_candidate",
        "rule_action_multi_candidate",
        "bertopic_topic",
        "strong_domains",
        "weak_domains",
        "app_packages_detected",
        "media_doc",
        "action_doc",
    ]
    + MEDIA_LABELS + ["media_unknown"]
    + ACTION_LABELS + ["action_unknown"]
)

result_cols = [c for c in result_cols if c in df.columns]

low_confidence_review = df[
    (df["final_media"] == "media_unknown")
    | (df["final_action"] == "action_unknown")
    | (df["media_source"] == "topic_high_confidence")
    | (df["action_source"] == "topic_high_confidence")
    | (df["rule_media_multi_candidate"] == 1)
    | (df["rule_action_multi_candidate"] == 1)
].copy()

weak_url_review = df[
    (df["weak_domains"].apply(lambda x: len(x) > 0))
    & (df["strong_domains"].apply(lambda x: len(x) == 0))
].copy()

topic_media_review_sorted = topic_media_review.sort_values(
    ["media_needs_review", "topic_size", "media_best_score"],
    ascending=[False, False, False],
)

topic_action_review_sorted = topic_action_review.sort_values(
    ["action_needs_review", "topic_size", "action_best_score"],
    ascending=[False, False, False],
)

print("low_confidence_review 행 수:", len(low_confidence_review))
print("weak_url_review 행 수:", len(weak_url_review))

display(low_confidence_review[result_cols].head(20))
display(weak_url_review[result_cols].head(20))


## 22. 결과 저장

이 셀은 최종 결과와 검수용 파일을 저장합니다.

media_action_flag_result.csv는 최종 라벨과 one-hot flag가 들어간 결과 파일입니다.

media_topic_review.csv는 media topic 후보를 검수하기 위한 파일입니다.

action_topic_review.csv는 action topic 후보를 검수하기 위한 파일입니다.

low_confidence_review.csv는 우선 검수해야 할 행만 모은 파일입니다.

weak_url_review.csv는 weak URL 때문에 오탐 가능성이 있는 행을 모은 파일입니다.

validation_labeling_sample.xlsx는 사람이 true_media와 true_action을 직접 채울 검수 샘플입니다.

media_action_flag_result_with_review.xlsx는 위 내용을 여러 시트로 묶은 엑셀 파일입니다.


In [ ]:
result_path = OUT_DIR / "media_action_flag_result.csv"
topic_media_path = OUT_DIR / "media_topic_review.csv"
topic_action_path = OUT_DIR / "action_topic_review.csv"
low_conf_path = OUT_DIR / "low_confidence_review.csv"
weak_url_path = OUT_DIR / "weak_url_review.csv"
validation_path = OUT_DIR / "validation_labeling_sample.xlsx"
excel_path = OUT_DIR / "media_action_flag_result_with_review.xlsx"

df[result_cols].to_csv(result_path, index=False, encoding="utf-8-sig")
topic_media_review_sorted.to_csv(topic_media_path, index=False, encoding="utf-8-sig")
topic_action_review_sorted.to_csv(topic_action_path, index=False, encoding="utf-8-sig")
low_confidence_review[result_cols].head(MAX_REVIEW_ROWS).to_csv(low_conf_path, index=False, encoding="utf-8-sig")
weak_url_review[result_cols].head(MAX_REVIEW_ROWS).to_csv(weak_url_path, index=False, encoding="utf-8-sig")
validation_sample[validation_cols].to_excel(validation_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df[result_cols].head(MAX_REVIEW_ROWS).to_excel(writer, sheet_name="result_sample", index=False)
    topic_media_review_sorted.head(MAX_REVIEW_ROWS).to_excel(writer, sheet_name="media_topic_review", index=False)
    topic_action_review_sorted.head(MAX_REVIEW_ROWS).to_excel(writer, sheet_name="action_topic_review", index=False)
    low_confidence_review[result_cols].head(MAX_REVIEW_ROWS).to_excel(writer, sheet_name="low_confidence", index=False)
    weak_url_review[result_cols].head(MAX_REVIEW_ROWS).to_excel(writer, sheet_name="weak_url_review", index=False)
    validation_sample[validation_cols].to_excel(writer, sheet_name="validation_sample", index=False)

print("저장 완료")
print("result_path:", result_path.resolve())
print("topic_media_path:", topic_media_path.resolve())
print("topic_action_path:", topic_action_path.resolve())
print("low_conf_path:", low_conf_path.resolve())
print("weak_url_path:", weak_url_path.resolve())
print("validation_path:", validation_path.resolve())
print("excel_path:", excel_path.resolve())


## 23. 결과가 여전히 좋지 않을 때 점검 순서

1. URL 도메인이 잘 추출됐는지 봅니다. strong_domains가 대부분 비어 있다면 원본 데이터에 URL이 없거나 URL 컬럼을 빠뜨린 것입니다.

2. rule_media_evidence와 rule_action_evidence를 확인합니다. evidence가 이상하면 taxonomy 키워드가 너무 넓거나 잘못된 것입니다.

3. action_click이 과도하게 많다면 click, 방문, 접속 같은 단어가 너무 넓게 쓰이고 있는지 확인합니다. 클릭 로그가 있다는 이유로 action_click을 붙이면 안 됩니다.

4. media_unknown이 많다면 taxonomy에 실제 광고 문구를 더 추가해야 합니다. unknown 샘플을 보고 반복적으로 키워드를 보강하세요.

5. topic으로 채운 라벨이 많이 틀린다면 USE_TOPIC_TO_FILL_UNKNOWN을 False로 두고 rule 기반 결과만 먼저 사용하세요.

6. 반드시 validation_labeling_sample.xlsx에 사람이 정답을 채워서 precision과 recall을 확인하세요. 정답셋이 없으면 성능을 객관적으로 판단할 수 없습니다.


In [ ]:
# Colab의 files.download()는 VSCode에서 동작하지 않습니다.
# 결과 파일은 OUT_DIR 폴더에 저장되어 있습니다.
# 아래에서 저장된 파일 목록을 확인하세요.

print('저장된 결과 파일 목록:')
for f in sorted(OUT_DIR.iterdir()):
    print(' ', f.name)


In [1]:
import os
from pathlib import Path

p = Path(r'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/eda_ready_new')
total = sum(f.stat().st_size for f in p.iterdir() if f.is_file())
print(f"{total / 1024**3:.2f} GB")
for f in sorted(p.iterdir()):
    print(f"  {f.name}: {f.stat().st_size / 1024**2:.1f} MB")

1.18 GB
  ad_attr_map.parquet: 9.3 MB
  ad_mart_v5.parquet: 0.9 MB
  ad_master_clean.parquet: 32.7 MB
  ad_outcome.parquet: 5.6 MB
  ads_join_info_labeled.parquet: 740.6 MB
  finance_clean1.parquet: 84.2 MB
  finance_clean2.parquet: 78.9 MB
  hourly_report.parquet: 34.9 MB
  ive_ad_classification.parquet: 13.0 MB
  main_funnel.parquet: 155.6 MB
  model1_test.parquet: 0.2 MB
  model1_train.parquet: 0.3 MB
  model1_val.parquet: 0.2 MB
  model2_test.parquet: 0.2 MB
  model2_train.parquet: 0.4 MB
  model2_val.parquet: 0.2 MB
  preprocessing_pipeline.joblib: 0.1 MB
  sched_clean.parquet: 7.7 MB
  user_daily_activity_clean.parquet: 42.0 MB


In [2]:
import os
from pathlib import Path

for folder in ['parquet_debug', 'parquet_debug01', 'eda_ready_new']:
    p = Path(rf'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/{folder}')
    total = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
    print(f"{folder}: {total / 1024**3:.2f} GB")

parquet_debug: 0.84 GB
parquet_debug01: 1.05 GB
eda_ready_new: 1.18 GB


In [3]:
from pathlib import Path

p = Path(r'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/parquet_debug01')
for f in sorted(p.iterdir()):
    print(f.name)

ads_join_info_labeled.parquet
ads_outcome.parquet
finance_clean1.parquet
finance_clean2.parquet
ive_ad_classification.parquet
main_funnel.parquet
sched.parquet
tb_user_daily_activity.parquet
